# LongFlow — clean-frames ablation (the closed-loop regression diagnosis)

Runtime: **A100 GPU**, ~1–1.5 h. Pre-registered criteria: NOTES.md
"CLEAN-FRAMES ABLATION PRE-REGISTRATION" (2026-08-18).

Question under test: did capture v2's **47% noised-feedback frames** cause
the closed-loop identity regression (HV2 close-out verdict 3: control head
on v2 data scored WER 0.116 / sim 0.21–0.30 in the loop, vs the July
v1-data head's 0.031 / 0.52)? One arm: control architecture, trained ONLY
on σ=0 frames (~268K pairs), same recipe, same held-out split, then the
same closed-loop protocol at 2 seeds.

| cell | what |
|---|---|
| 1 | cold start (identical to train_headv2 cell 1) |
| 2 | filter + split + CLEAN-FRAME pool (sigma_bucket == 0 only) |
| 3 | train `cleanabl_20k` (20K, ckpts every 5K) |
| 4 | held-out teacher-forced at 20K (heun8+CFG, n=25) |
| 5 | closed loop: 2 seeds, heun8+CFG, GN5–8 script |
| 6 | bundle → Drive root `cleanabl_eval.zip` |


In [ ]:
# ===== COLD START (idempotent) — run me first, wait for READY =====
NOTEBOOK_VERSION = "Clean-frames ablation v1.0 (2026-08-18)"
print(f"*** {NOTEBOOK_VERSION} ***")
%cd /content
!git clone -q https://github.com/vibevoice-community/VibeVoice.git 2>/dev/null || true
%cd /content/VibeVoice
!git checkout -q 07cb79fea
!pip install -q -e .

import torch
from vibevoice.modular.modeling_vibevoice_inference import (
    VibeVoiceForConditionalGenerationInference,
)
from vibevoice.processor.vibevoice_processor import VibeVoiceProcessor
MODEL_ID = "microsoft/VibeVoice-1.5B"
model = VibeVoiceForConditionalGenerationInference.from_pretrained(
    MODEL_ID, torch_dtype=torch.bfloat16, device_map="cuda"
)
processor = VibeVoiceProcessor.from_pretrained(MODEL_ID)
model.eval()

from google.colab import drive
drive.mount("/content/drive")
import glob, json, os, shutil, sys, time
from pathlib import Path
import numpy as np
import soundfile as sf

if os.path.exists("/content/LongFlow/src"):
    !cd /content/LongFlow && git pull -q
else:
    !git clone -q https://github.com/Josh-E-S/LongFlow.git /content/LongFlow || true
assert os.path.exists("/content/LongFlow/src"), "clone failed — check repo access"
sys.path.insert(0, "/content/LongFlow")
!cd /content/LongFlow && git log --oneline -1

from src.cache.capture import load_utterance
from src.flow_head.cfm import heun_sample
from src.flow_head.integration import CFGFlowHeadPatch, _CFGField
from src.flow_head.model import FlowHead, FlowHeadConfig
from src.flow_head.trainer import (
    PairData, filter_flagged, load_checkpoint, pairs_from_files, train,
)

CACHE_V2_DRIVE = "/content/drive/MyDrive/longflow_p1_cache_v2"
CKPT_DIR = "/content/drive/MyDrive/longflow_p1_ckpt"
TRAIN_CACHE_V1 = "/content/drive/MyDrive/longflow_p1_cache"
EVAL_CACHE_DIR = "/content/drive/MyDrive/longflow_p1_evalcache"
GATE3_DIR = "/content/drive/MyDrive/longflow_gate3"
OUT = "/content/cleanabl"
DRIVE_OUT = "/content/drive/MyDrive/longflow_cleanabl"
EVAL_DIR = "/content/cleanabl_eval"
for d in (OUT, DRIVE_OUT, EVAL_DIR):
    os.makedirs(d, exist_ok=True)

LOCAL_CACHE = "/content/cache_v2"
if not os.path.exists(LOCAL_CACHE) or len(glob.glob(f"{LOCAL_CACHE}/*.pt")) < 240:
    os.makedirs(LOCAL_CACHE, exist_ok=True)
    print("bulk-copying cache v2 to local disk (~4 GB, a few minutes)...", flush=True)
    !cp {CACHE_V2_DRIVE}/*.pt {LOCAL_CACHE}/
print(f"{len(glob.glob(f'{LOCAL_CACHE}/*.pt'))} cache files local")

if os.path.exists(f"{DRIVE_OUT}/cleanabl_report.json"):
    with open(f"{DRIVE_OUT}/cleanabl_report.json") as f:
        report = json.load(f)
    print(f"resuming: {len(report['runs'])} runs already recorded")
else:
    report = {"notebook_version": NOTEBOOK_VERSION, "runs": []}

def done(tag):
    return os.path.exists(f"{DRIVE_OUT}/{tag}.wav")

def save_wav(tag, wav, meta):
    sf.write(f"{OUT}/{tag}.wav", wav, 24000)
    shutil.copy(f"{OUT}/{tag}.wav", f"{DRIVE_OUT}/{tag}.wav")
    report["runs"] = [r for r in report["runs"] if r.get("tag") != tag] + [{"tag": tag, **meta}]
    with open(f"{DRIVE_OUT}/cleanabl_report.json", "w") as f:
        json.dump(report, f, indent=2)
    print(f"saved {tag}: {len(wav)/24000:.1f}s  {meta}", flush=True)

def gen_inputs(texts, prompt_lists):
    inputs = processor(text=texts, voice_samples=prompt_lists,
                       return_tensors="pt", padding=True)
    return {k: (v.to("cuda") if hasattr(v, "to") else v) for k, v in inputs.items()}

def decode_latents(z, chunk_frames=225):
    sc = model.model.speech_scaling_factor
    bi = model.model.speech_bias_factor
    z = z.to("cuda", torch.bfloat16)
    z = z / sc - bi
    wavs, shape_fn = [], None
    for i in range(0, z.shape[0], chunk_frames):
        chunk = z[i : i + chunk_frames]
        candidates = [shape_fn] if shape_fn is not None else [
            lambda c: c.unsqueeze(0), lambda c: c.unsqueeze(0).transpose(1, 2)
        ]
        decoded = None
        for fn in candidates:
            try:
                out = model.model.acoustic_tokenizer.decode(fn(chunk))
                decoded = out[0] if isinstance(out, tuple) else out
                shape_fn = fn
                break
            except Exception as e:
                print(f"decode attempt {tuple(fn(chunk).shape)} failed: {repr(e)[:150]}")
        if decoded is None:
            raise RuntimeError("both decode shapes failed — paste the errors to Claude")
        wavs.append(decoded.detach().float().cpu().numpy().squeeze())
        del out, decoded
        torch.cuda.empty_cache()
    return np.concatenate(wavs) if len(wavs) > 1 else wavs[0]

print("READY")


In [ ]:
# ===== Same file pool/split as HV2, then keep ONLY sigma==0 frames =====
HELD_OUT_PER_BIN = 5
FLAGS = "/content/LongFlow/experiments/p1_flow_head/capture_v2_audit_flags.json"

all_files = sorted(glob.glob(f"{LOCAL_CACHE}/*.pt"))
clean_files = filter_flagged(all_files, FLAGS)

def fname_bin(path):
    return int(Path(path).stem.split("_")[1].rstrip("w"))

by_bin = {}
for f in clean_files:
    by_bin.setdefault(fname_bin(f), []).append(f)
held_out_by_bin = {b: fs[:HELD_OUT_PER_BIN] for b, fs in sorted(by_bin.items())}
held_out_files = [f for fs in held_out_by_bin.values() for f in fs]
train_files = [f for b, fs in sorted(by_bin.items()) for f in fs[HELD_OUT_PER_BIN:]]
print(f"held-out {len(held_out_files)} / train {len(train_files)} scripts (identical to HV2 split)")

full = pairs_from_files(train_files, dual_stream=True)  # dual load to get sigma, then mask
mask = full.sigma_bucket == 0
raw_latent = full.latent * full.std + full.mean  # undo full-pool standardization
clean_latent = raw_latent[mask]
mean = clean_latent.mean(dim=0)
std = clean_latent.std(dim=0).clamp_min(1e-4)  # stats recomputed on the CLEAN subset
data = PairData(hidden=full.hidden[mask], latent=(clean_latent - mean) / std,
                mean=mean, std=std)
print(f"clean pool: {int(mask.sum())}/{full.hidden.shape[0]} frames "
      f"({100*float(mask.float().mean()):.1f}%)  d_model={data.d_model}  d_latent={data.d_latent}")
del full, raw_latent, clean_latent


In [ ]:
# ===== Train control architecture on clean frames: 20K, ckpts every 5K =====
TAG = "cleanabl_20k"
final = f"{CKPT_DIR}/{TAG}_step20000.pt"
if os.path.exists(final):
    print(f"{TAG}: final checkpoint already on Drive — skipping")
else:
    head = FlowHead(FlowHeadConfig(d_model=data.d_model, d_latent=data.d_latent))
    print(f"head: {head.param_count()/1e6:.2f}M params (control architecture)")
    t0 = time.time()
    train(head, data, steps=20000, batch_size=1024, lr=2e-4, lr_final=2e-5,
          ema_decay=0.9999, device="cuda", log_every=1000,
          checkpoint_every=5000,
          checkpoint_path_fn=lambda s: f"{CKPT_DIR}/{TAG}_step{s}.pt")
    print(f"done in {(time.time()-t0)/60:.1f} min")
    del head
    torch.cuda.empty_cache()


In [ ]:
# ===== Held-out teacher-forced at 20K (heun8 + CFG 1.3, n=25) =====
manifest = {"held_out_per_bin": HELD_OUT_PER_BIN, "teacher": {}, "checkpoints": {}}
head20, mean20, std20 = load_checkpoint(f"{CKPT_DIR}/cleanabl_20k_step20000.pt")
head20 = head20.to("cuda")

entries = []
for fpath in held_out_files:
    utt = load_utterance(fpath)
    tname = f"{utt.utt_id}_teacher.wav"
    if not os.path.exists(f"{EVAL_DIR}/{tname}"):
        sf.write(f"{EVAL_DIR}/{tname}", decode_latents(utt.latent.float()), 24000)
    manifest["teacher"][utt.utt_id] = {"audio": tname, "text": utt.text,
                                       "target_words": fname_bin(fpath)}
    name = f"{utt.utt_id}_step20000_C.wav"
    if not os.path.exists(f"{EVAL_DIR}/{name}"):
        field = _CFGField(head20, utt.neg_hidden.float().cuda(), 1.3)
        g = torch.Generator(device="cuda").manual_seed(0)
        z = heun_sample(field, utt.hidden.float().cuda(), head20.cfg.d_latent,
                        nfe=8, sway=0.0, generator=g)
        z = z * std20.cuda() + mean20.cuda()
        sf.write(f"{EVAL_DIR}/{name}", decode_latents(z), 24000)
    entries.append({"utt_id": utt.utt_id, "audio": name,
                    "teacher_audio": tname, "text": utt.text,
                    "target_words": fname_bin(fpath), "arm": "C"})
    print(f"rendered {utt.utt_id}", flush=True)
manifest["checkpoints"]["20000:C"] = entries
with open(f"{EVAL_DIR}/manifest.json", "w") as f:
    json.dump(manifest, f, indent=2)
print("held-out eval rendered")


In [ ]:
# ===== Closed loop: 2 seeds, heun8+CFG, same GN5–8 script =====
def drive_glob(pattern, tries=4, wait=15):
    for i in range(tries):
        hits = sorted(glob.glob(pattern))
        if hits:
            return hits
        print(f"empty listing for {pattern} — retry {i+1}/{tries} in {wait}s", flush=True)
        time.sleep(wait)
    raise RuntimeError(f"still empty after {tries} tries: {pattern}")

V1_LOCAL = "/content/v1texts"
if len(glob.glob(f"{V1_LOCAL}/*.pt")) < 300:
    os.makedirs(V1_LOCAL, exist_ok=True)
    for f in drive_glob(f"{TRAIN_CACHE_V1}/*.pt")[-300:]:
        shutil.copy(f, V1_LOCAL)
sents = []
for f in sorted(glob.glob(f"{V1_LOCAL}/*.pt")):
    d = torch.load(f, weights_only=True)
    sents.append(d["text"].strip().rstrip(".") + ".")
pool = sents[:200]
P0 = drive_glob(f"{EVAL_CACHE_DIR}/*_prompt.wav")[0]

def turnscript(sentences, target=60, speaker=1):
    turns, cur, w = [], [], 0
    for s in sentences:
        cur.append(s); w += len(s.split())
        if w >= target:
            turns.append(f"Speaker {speaker}: " + " ".join(cur)); cur, w = [], 0
    if cur:
        turns.append(f"Speaker {speaker}: " + " ".join(cur))
    return "\n".join(turns) + "\n"

ABL_WORDS, w = [], 0
for s in pool:
    ABL_WORDS.append(s); w += len(s.split())
    if w >= 800:
        break
ABL_SCRIPT = turnscript(ABL_WORDS)
print("closed-loop script:", w, "words")
report["cl_script"] = ABL_SCRIPT
report["cl_words"] = w

for seed in (0, 1):
    tag = f"cleanabl_cfg_heun8_s{seed}"
    if done(tag):
        print(f"{tag}: already on Drive — skipping")
        continue
    torch.manual_seed(seed)
    with CFGFlowHeadPatch(model, head20, mean20, std20, nfe=8, sway=0.0,
                          sampler=heun_sample) as patch, torch.inference_mode():
        gen = model.generate(**gen_inputs([ABL_SCRIPT], [[P0]]),
                             tokenizer=processor.tokenizer,
                             cfg_scale=1.3, max_new_tokens=3000)
    wav = gen.speech_outputs[0].detach().float().cpu().numpy().squeeze()
    zs = torch.cat(patch.latents) if patch.latents else torch.zeros(1)
    save_wav(tag, wav, {"seed": seed, "frames": patch.calls,
                        "latent_std": round(float(zs.std()), 3)})


In [ ]:
# ===== Bundle → Drive root =====
import zipfile
teacher_ref = f"{GATE3_DIR}/t1_turnsplit_p0.wav"
assert os.path.exists(teacher_ref), "GN3 teacher reference missing from Drive"
shutil.copy(teacher_ref, f"{EVAL_DIR}/t1_turnsplit_p0.wav")
with open(f"{EVAL_DIR}/cleanabl_report.json", "w") as f:
    json.dump(report, f, indent=2)

ZIP = "/content/drive/MyDrive/cleanabl_eval.zip"
with zipfile.ZipFile(ZIP, "w") as z:
    for f in os.listdir(EVAL_DIR):
        z.write(f"{EVAL_DIR}/{f}", f)
    for f in os.listdir(DRIVE_OUT):
        if f.endswith(".wav"):
            z.write(f"{DRIVE_OUT}/{f}", f"closed_loop/{f}")
print(f"bundle at {ZIP} ({os.path.getsize(ZIP)/1e9:.2f} GB) — run score_cleanabl_gpu_colab.ipynb next")
